#**Installation Library**

In [ ]:
!pip install pandas dask[dataframe] psutil
!pip install polars

**TEST CODE**

In [ ]:
import asyncio
import aiohttp
import csv
import os
import time
from datetime import UTC, datetime

# ==========================================
# --- TEST CONFIGURATION ---
# ==========================================
CATEGORY = "corporate"
LIMIT_PER_PAGE = 20

# ONLY SCRAPE 2 PAGES FOR TESTING
START_PAGE = 1
END_PAGE = 2
BATCH_SIZE = 2
# ==========================================

def convert_timestamp(ms):
    if not ms: return ""
    try: return datetime.fromtimestamp(ms / 1000, tz=UTC).strftime('%Y-%m-%d %H:%M:%S')
    except Exception: return ""

async def fetch_articles(session, offset, category, limit):
    url = f"https://theedgemalaysia.com/api/loadMoreCategories?offset={offset}&categories={category}&limit={limit}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://theedgemalaysia.com/"
    }

    try:
        async with session.get(url, headers=headers, timeout=15) as response:
            if response.status == 200:
                data = await response.json()
                return data.get("results", [])
    except Exception as e:
        print(f"Error fetching offset {offset}: {e}")
    return []

def save_to_csv(data, filename):
    fieldnames = [
        "category", "sub-category", "title", "author",
        "source", "summary", "link", "created date", "updated date"
    ]
    file_exists = os.path.isfile(filename)

    with open(filename, "w", newline="", encoding="utf-8") as csvfile: # Changed to 'w' (overwrite) for testing
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(data)

    print(f"💾 Saved {len(data)} test articles to {filename}")

async def main():
    output_filename = f"TEST_theedge_output.csv"
    seen_titles = set()

    print(f"🛠️ Starting TEST RUN for Pages {START_PAGE} to {END_PAGE}...")

    async with aiohttp.ClientSession() as session:
        start_index = START_PAGE - 1
        end_index = END_PAGE

        tasks = []
        for i in range(start_index, end_index):
            offset = i * LIMIT_PER_PAGE
            tasks.append(fetch_articles(session, offset, CATEGORY, LIMIT_PER_PAGE))

        batch_results = await asyncio.gather(*tasks)

        all_articles = []

        for page_articles in batch_results:
            if not page_articles: continue

            for item in page_articles:
                title = item.get("title", "").strip()
                if title in seen_titles: continue
                seen_titles.add(title)

               # --- LINK GENERATION LOGIC ---
                alias_path = item.get("alias", "")
                # We attach it directly because alias already contains "node/..."
                full_link = f"https://theedgemalaysia.com/{alias_path}" if alias_path else ""

                article_data = {
                    "category": item.get("category", ""),
                    "sub-category": item.get("flash", ""),
                    "title": title,
                    "author": item.get("author", ""),
                    "source": item.get("source", ""),
                    "summary": item.get("summary", ""),
                    "link": full_link,
                    "created date": convert_timestamp(item.get("created", 0)),
                    "updated date": convert_timestamp(item.get("updated", 0))
                }

                # Print the very first article to the terminal so you can test the link easily
                if len(all_articles) == 0:
                    print("\n--- 🔍 SNEAK PEEK OF FIRST ARTICLE ---")
                    for key, value in article_data.items():
                        print(f"{key.upper()}: {value}")
                    print("--------------------------------------\n")

                all_articles.append(article_data)

        if all_articles:
            save_to_csv(all_articles, output_filename)

    print(f"\n✅ Test Complete! Please check your terminal output and the {output_filename} file.")

# In Jupyter Notebooks, you can directly await the main function
await main()

#**CHECK API FOR LINK**

In [ ]:
import requests

# Pull just 1 article from the API
url = "https://theedgemalaysia.com/api/loadMoreCategories?offset=0&categories=corporate&limit=1"
headers = {"User-Agent": "Mozilla/5.0"}

response = requests.get(url, headers=headers)
data = response.json().get("results", [])

if data:
    raw_article = data[0]
    print("🔍 RAW API KEYS AVAILABLE:\n" + "-"*30)
    for key, value in raw_article.items():
        # Truncate long text so it's easy to read in your terminal
        print(f"Key: '{key}' ---> Value: {str(value)[:50]}...")
else:
    print("Failed to pull data.")

#**RUN WEB SCRAPPING**



In [ ]:
import asyncio
import aiohttp
import csv
import os
import time
from datetime import UTC, datetime

# ==========================================
# --- CONFIGURATION ---
# ==========================================
CATEGORIES_TO_SCRAPE = [
    "corporate",
    "economy",
    "malaysia",
    "world",
    "options",
    "city-country",
    "wealth",
    "digitaledge",
    "esg",
    "tech",
    "court",
    "politics"
]

LIMIT_PER_PAGE = 20
CONCURRENCY_LIMIT = 50
BATCH_SIZE = 100
# ==========================================

def convert_timestamp(ms):
    if not ms: return ""
    try: return datetime.fromtimestamp(ms / 1000, tz=UTC).strftime('%Y-%m-%d %H:%M:%S')
    except Exception: return ""

async def fetch_articles(session, offset, category, limit, semaphore):
    url = f"https://theedgemalaysia.com/api/loadMoreCategories?offset={offset}&categories={category}&limit={limit}"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
        "Referer": "https://theedgemalaysia.com/"
    }

    async with semaphore:
        for attempt in range(3):
            try:
                async with session.get(url, headers=headers, timeout=15) as response:
                    if response.status == 200:
                        data = await response.json()
                        return data.get("results", []) # Returns [] if no more data
                    elif response.status == 429:
                        await asyncio.sleep(2 ** attempt)
            except Exception:
                await asyncio.sleep(2 ** attempt)
        return None # Returns None ONLY if network completely failed

def save_to_csv(data, filename):
    fieldnames = [
        "category", "sub-category", "title", "author",
        "source", "summary", "link", "created date", "updated date"
    ]
    file_exists = os.path.isfile(filename)

    with open(filename, "a", newline="", encoding="utf-8") as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        if not file_exists:
            writer.writeheader()
        writer.writerows(data)

    print(f"💾 Saved {len(data)} articles to {filename}")

async def main():
    output_filename = "theedge_100k_master.csv"
    semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
    seen_titles = set()
    total_collected_overall = 0
    start_time = time.time()

    async with aiohttp.ClientSession() as session:

        # Loop through each category one by one
        for current_category in CATEGORIES_TO_SCRAPE:
            print(f"\n🚀 === STARTING NEW CATEGORY: {current_category.upper()} ===")

            # We will try to fetch up to 2000 pages (40,000 articles) per category
            max_pages_per_category = 2000
            category_exhausted = False

            for batch_start in range(0, max_pages_per_category, BATCH_SIZE):
                if category_exhausted:
                    break

                batch_end = batch_start + BATCH_SIZE
                print(f"⚡ Fetching Pages {batch_start + 1} to {batch_end} for '{current_category}'...")

                tasks = []
                for i in range(batch_start, batch_end):
                    offset = i * LIMIT_PER_PAGE
                    tasks.append(fetch_articles(session, offset, current_category, LIMIT_PER_PAGE, semaphore))

                batch_results = await asyncio.gather(*tasks)

                all_articles = []

                for page_articles in batch_results:
                    if page_articles is None:
                        # Network error on this page, just skip it and keep going!
                        continue

                    if len(page_articles) == 0:
                        # True end of data for this category
                        category_exhausted = True
                        continue

                    for item in page_articles:
                        title = item.get("title", "").strip()
                        if title in seen_titles:
                            continue
                        seen_titles.add(title)

                        alias_path = item.get("alias", "")
                        full_link = f"https://theedgemalaysia.com/{alias_path}" if alias_path else ""

                        all_articles.append({
                            "category": item.get("category", ""),
                            "sub-category": item.get("flash", ""),
                            "title": title,
                            "author": item.get("author", ""),
                            "source": item.get("source", ""),
                            "summary": item.get("summary", ""),
                            "link": full_link,
                            "created date": convert_timestamp(item.get("created", 0)),
                            "updated date": convert_timestamp(item.get("updated", 0))
                        })
                        total_collected_overall += 1

                if all_articles:
                    save_to_csv(all_articles, output_filename)

            print(f"🏁 Finished all available articles in '{current_category}'.")

    elapsed_time = time.time() - start_time
    print(f"\n✅ ALL CATEGORIES COMPLETE!")
    print(f"📈 Total Unique Records Collected: {total_collected_overall}")
    print(f"⏱️ Total Time: {elapsed_time:.2f} seconds.")
    if elapsed_time > 0:
        print(f"📊 Overall Throughput: {total_collected_overall / elapsed_time:.2f} records per second.")

# In Jupyter Notebooks, you can directly await the main function
await main()